# Critic-Actor Loop on HumanEval

Companion notebook for the [Multi-Agent Patterns in Depth](https://codemia.io/courses/applied_ai_systems/multi_agent_patterns_in_depth) lesson on Codemia.

**What you will build**: a two-role system for code generation. The **actor** writes a Python
solution to a HumanEval problem. The **critic** runs it against the problem's unit tests and
reports exactly which assertions failed and what the code returned. The actor revises with that
report, for up to three rounds. You will measure the pass rate against a single-shot baseline on
the full HumanEval set, plot accuracy against iteration count, and check whether the gain comes
from the feedback or just from extra attempts.

## Setup

The actor needs a model:

- With an `OPENAI_API_KEY` set, the actor is `gpt-4o-mini`. The full 164-problem run takes a few
  minutes and costs a few cents.
- Without one, the actor is `Qwen2.5-Coder-1.5B-Instruct` running locally. Use a GPU runtime in
  Colab (Runtime > Change runtime type > T4 GPU). On CPU the notebook drops to 20 problems so it
  finishes, and the numbers get noisy.

The critic needs no model at all. It is a test runner and a formatter, which is the point: the
most useful critic for code is the one that executes it.

In [ ]:
%pip install -q openai datasets matplotlib transformers accelerate

In [ ]:
import json
import os
import re
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt

try:
    from google.colab import userdata
    os.environ.setdefault('OPENAI_API_KEY', userdata.get('OPENAI_API_KEY') or '')
except Exception:
    pass

BACKEND = 'openai' if os.environ.get('OPENAI_API_KEY') else 'local'
MAX_ROUNDS = 3          # revisions after the first attempt
print('actor backend:', BACKEND)

In [ ]:
from datasets import load_dataset

problems = list(load_dataset('openai/openai_humaneval', split='test'))
if BACKEND == 'local':
    import torch
    if not torch.cuda.is_available():
        problems = problems[:20]  # CPU: keep it to a coffee break, not an afternoon
print(len(problems), 'problems')
print(problems[0]['prompt'])

## Concept recap

Self-refinement with a model as its own critic tends to stall: the model that wrote a bug is
bad at spotting it, and "looks correct to me" is the usual verdict. Splitting the roles helps
only if the critic has information the actor lacks. For code that information is execution.
A critic that says *`assert candidate([1, 2, 3]) == 6` failed, the function returned 5* hands
the actor a concrete counterexample, and a counterexample is much easier to fix than a vague
"this seems wrong".

## The critic

Each candidate runs in a separate Python process with a timeout, because model-written code can
loop forever or call `exit()`. Inside that process the harness runs `check()` one statement at a
time, so one failing assertion does not hide the others. For `assert f(x) == y` it evaluates both
sides and reports what the function actually returned.

In [ ]:
HARNESS = r'''
import ast, json, sys

src, test, entry = json.load(sys.stdin)
ns = {}
try:
    exec(src, ns)
except BaseException as e:
    print(json.dumps({'passed': False, 'n_tests': 0,
                      'failures': [f'code did not load: {type(e).__name__}: {e}']}))
    sys.exit()
env = {}
exec(test, env)
check = next(n for n in ast.parse(test).body if isinstance(n, ast.FunctionDef) and n.name == 'check')
env[check.args.args[0].arg] = ns.get(entry)
failures, n_tests = [], 0
for stmt in check.body:
    seg = ast.get_source_segment(test, stmt).strip()
    is_test = any(isinstance(n, ast.Assert) for n in ast.walk(stmt))
    n_tests += is_test
    try:
        exec(compile(ast.Module([stmt], []), '<test>', 'exec'), env)
    except AssertionError:
        detail = seg[:200]
        t = getattr(stmt, 'test', None)
        if isinstance(t, ast.Compare) and len(t.ops) == 1:
            try:
                got = eval(compile(ast.Expression(t.left), '<got>', 'eval'), env)
                want = eval(compile(ast.Expression(t.comparators[0]), '<want>', 'eval'), env)
                detail += f'\n    got {got!r:.120}, expected {want!r:.120}'
            except BaseException:
                pass
        failures.append(detail)
    except BaseException as e:
        failures.append(f'{seg[:200]}\n    raised {type(e).__name__}: {str(e)[:120]}')
print(json.dumps({'passed': not failures, 'n_tests': n_tests, 'failures': failures}))
'''

In [ ]:
def critic(problem, code, timeout=10):
    payload = json.dumps([code, problem['test'], problem['entry_point']])
    try:
        proc = subprocess.run([sys.executable, '-c', HARNESS], input=payload,
                              capture_output=True, text=True, timeout=timeout)
        return json.loads(proc.stdout.strip().splitlines()[-1])
    except subprocess.TimeoutExpired:
        return {'passed': False, 'n_tests': 0, 'failures': [f'timed out after {timeout}s']}
    except Exception as e:  # crashed before printing a verdict
        return {'passed': False, 'n_tests': 0, 'failures': [f'harness error: {e}']}


def critique_text(verdict, limit=3):
    lines = [f'{len(verdict["failures"])} of {verdict["n_tests"] or "?"} checks failed:']
    for f in verdict['failures'][:limit]:
        lines.append('- ' + f)
    return '\n'.join(lines)

Test the critic before trusting it: the reference solution must pass, and a deliberately broken
one must fail with a readable report.

In [ ]:
p = problems[0]
print('canonical:', critic(p, p['prompt'] + p['canonical_solution']))
broken = p['prompt'] + '    return False\n'
print(critique_text(critic(p, broken)))

## The actor

The actor gets the HumanEval prompt (signature plus docstring) and must return the whole function
in a code block. On a revision it also sees its previous code and the critic's report.

The program that runs is `prompt + model code`. The prompt alone is valid Python, a function whose
body is its docstring, so any imports or helper functions it defines stay in scope, and the
model's definition simply replaces the stub.

In [ ]:
SYSTEM = 'You are an expert Python programmer. Reply with one ```python code block containing the complete function, and nothing else.'


def first_prompt(problem):
    return f'Complete this function.\n\n```python\n{problem["prompt"]}\n```'


def revise_prompt(problem, code, verdict):
    return (f'Your solution to this function failed its tests.\n\n```python\n{problem["prompt"]}\n```'
            f'\n\nYour code:\n```python\n{code}\n```\n\nTest report:\n{critique_text(verdict)}'
            '\n\nFix the function.')


def extract_code(reply):
    blocks = re.findall(r'```(?:python)?\n(.*?)```', reply, re.S)
    return blocks[0] if blocks else reply

In [ ]:
_local = {}


def generate(prompt, temperature=0.0):
    msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': prompt}]
    if BACKEND == 'openai':
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model='gpt-4o-mini', messages=msgs,
                                             temperature=temperature, max_tokens=800)
        return r.choices[0].message.content
    if not _local:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        name = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
        _local['tok'] = AutoTokenizer.from_pretrained(name)
        _local['model'] = AutoModelForCausalLM.from_pretrained(
            name, torch_dtype='auto', device_map='auto')
    tok, model = _local['tok'], _local['model']
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt',
                                  return_dict=True).to(model.device)
    sample = dict(do_sample=True, temperature=temperature) if temperature > 0 else dict(do_sample=False)
    out = model.generate(**enc, max_new_tokens=600, pad_token_id=tok.eos_token_id, **sample)
    return tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)

## The loop

`solve()` returns the round at which a problem first passed (0 is the single-shot attempt), or
`None` if it never did. With `feedback=False` it is the control condition: same number of
attempts, but each one is a fresh sample that never sees the critic's report.

In [ ]:
def solve(problem, rounds=MAX_ROUNDS, feedback=True):
    code = extract_code(generate(first_prompt(problem)))
    verdict = critic(problem, problem['prompt'] + '\n' + code)
    history = [(code, verdict)]
    for r in range(1, rounds + 1):
        if verdict['passed']:
            return r - 1, history
        if feedback:
            reply = generate(revise_prompt(problem, code, verdict))
        else:
            reply = generate(first_prompt(problem), temperature=0.8)
        code = extract_code(reply)
        verdict = critic(problem, problem['prompt'] + '\n' + code)
        history.append((code, verdict))
    return (rounds if verdict['passed'] else None), history


def run_all(feedback=True):
    workers = 8 if BACKEND == 'openai' else 1  # a local model serves one request at a time
    with ThreadPoolExecutor(workers) as pool:
        return list(pool.map(lambda p: solve(p, feedback=feedback), problems))

## Experiment

Run every problem through the critic-actor loop. The single-shot baseline is free: it is round 0
of the same run, the first attempt before any feedback.

In [ ]:
results = run_all(feedback=True)


def curve(results):
    n = len(results)
    return [sum(r is not None and r <= k for r, _ in results) / n for k in range(MAX_ROUNDS + 1)]


with_feedback = curve(results)
for k, acc in enumerate(with_feedback):
    label = 'single-shot baseline' if k == 0 else f'after {k} revision(s)'
    print(f'{label:<22} pass rate {acc:6.1%}')

One problem the loop rescued, start to finish: the failing first attempt, the critic's report,
and the code that passed.

In [ ]:
rescued = [(p, h) for p, (r, h) in zip(problems, results) if r and r > 0]
print(len(rescued), 'problems failed first and passed after feedback')
if rescued:
    p, h = rescued[0]
    print('\n###', p['task_id'], '\n--- first attempt\n' + h[0][0])
    print('--- critic\n' + critique_text(h[0][1]))
    print('--- passing revision\n' + h[-1][0])

## Variations

**Is it the feedback, or just more tries?** Re-run with the same budget of attempts, but each
retry is a fresh sample at temperature 0.8 that never sees the test report. If the two curves
end in the same place, the critic is not doing anything a dice roll would not.

In [ ]:
control = curve(run_all(feedback=False))
rounds = list(range(MAX_ROUNDS + 1))
plt.figure(figsize=(6, 4))
plt.plot(rounds, with_feedback, 'o-', label='critic feedback')
plt.plot(rounds, control, 's--', label='resample, no feedback')
plt.axhline(with_feedback[0], color='grey', lw=0.8, ls=':', label='single-shot')
plt.xticks(rounds); plt.xlabel('attempt (0 = first try)'); plt.ylabel('pass rate')
plt.title(f'HumanEval, {len(problems)} problems, actor = {BACKEND}')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print('final pass rate  feedback: %.1f%%   resample: %.1f%%' % (100 * with_feedback[-1], 100 * control[-1]))

In [ ]:
# Where does the loop give up? The failures that survive every round, grouped by what the
# critic last saw. Timeouts and load errors are a different problem from wrong answers.
stuck = [h[-1][1]['failures'][0] for r, h in results if r is None]
kinds = {'timeout': 0, 'did not load': 0, 'raised': 0, 'wrong answer': 0}
for f in stuck:
    key = ('timeout' if 'timed out' in f else 'did not load' if 'did not load' in f
           else 'raised' if 'raised' in f else 'wrong answer')
    kinds[key] += 1
print(len(stuck), 'never passed:', kinds)

Things worth noticing:

1. **The baseline sets the ceiling on the gain.** The lesson's 65% to 80-85% comes from a weaker
   actor. A stronger actor starts higher, so it has fewer failures for the critic to work on,
   and the absolute gain shrinks even when the loop is working.
2. **Most of the gain arrives in the first revision.** If the actor cannot fix a bug after
   seeing a concrete counterexample, a second or third look rarely helps. That makes the round
   budget a cost decision with a fast-flattening curve.
3. **This critic is allowed to see the grading tests.** That is fine for studying the loop, and
   it is an upper bound: in production the hidden tests do not exist. Real systems run the
   docstring examples, tests the model generated itself, or a type checker, and gain less.

## What to try next

- Replace the critic's input with only the docstring examples (parse the `>>>` lines) and keep
  the real tests for grading. The gap between the two curves is how much the critic was leaning
  on test leakage.
- Make the critic a model: give it the code and the test report and ask for a one-paragraph
  diagnosis instead of passing the raw report. Measure whether the diagnosis beats the raw
  failures, and what it costs.
- Swap actors. Run the local 1.5B model and `gpt-4o-mini` through the same loop and compare how
  much each gains from feedback, not just where each starts.